# EDA — CHB-MIT Dataset

Exploratory data analysis for the SeizureHorizon project.  
Dataset: [CHB-MIT Scalp EEG Database](https://physionet.org/content/chbmit/1.0.0/)  
Goal: Understand pre-ictal EEG characteristics before building the prediction pipeline.

---
## 1. Load Raw EEG Sample

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch

# Suppress MNE's verbose output for cleaner notebook
mne.set_log_level('WARNING')

In [ ]:
# --- Load a CHB-MIT .edf file ---
EDF_PATH = '../data/raw/chb-mit/chb01/chb01_03.edf'

raw = mne.io.read_raw_edf(EDF_PATH, preload=True, verbose=False)

print('=== Recording Info ===')
print(f'Duration      : {raw.times[-1]:.1f} seconds ({raw.times[-1]/60:.1f} minutes)')
print(f'Sampling rate : {raw.info["sfreq"]} Hz')
print(f'Channels      : {len(raw.ch_names)}')
print()
print('Channel names:')
print(raw.ch_names)

In [ ]:
# --- Quick visual inspection of raw signal (first 10 seconds) ---
raw.plot(
    duration=10,
    n_channels=10,
    scalings='auto',
    title='Raw EEG — first 10 seconds (unfiltered)',
    show=True,
    block=True
)

---
## 2. Preprocessing

Steps applied:
- **Bandpass filter** 0.5–40 Hz — removes DC drift and high-frequency noise
- **Notch filter** at 60 Hz — removes US powerline interference
- **Channel selection** — keep only standard 10-20 EEG channels
- **Crop to pre-ictal window** — 60 to 30 minutes before seizure onset

In [ ]:
# --- Bandpass and notch filtering ---
raw.filter(0.5, 40.0, fir_design='firwin', verbose=False)
raw.notch_filter(60.0, verbose=False)

print('Filtering applied: bandpass 0.5–40 Hz, notch 60 Hz')

In [ ]:
# --- Keep only standard 10-20 EEG channels ---
# CHB-MIT channels are formatted as 'FP1-F7', 'F7-T7', etc. (bipolar montage)
# We keep all available channels for now and inspect
print(f'Channels after filtering: {len(raw.ch_names)}')
print(raw.ch_names)

In [ ]:
# --- Crop to pre-ictal window ---
# Seizure onset for chb01_03.edf is at 2996 seconds
# Source: chb01-summary.txt in the CHB-MIT dataset
# Update this value for other recordings
SEIZURE_ONSET = 2996   # seconds — from chbXX-summary.txt
HORIZON       = 3600   # prediction horizon: 60 minutes before onset
WINDOW_END    = 1800   # end of pre-ictal window: 30 minutes before onset

window_start = max(0, SEIZURE_ONSET - HORIZON)
window_end   = SEIZURE_ONSET - WINDOW_END

print(f'Seizure onset  : {SEIZURE_ONSET}s ({SEIZURE_ONSET/60:.1f} min)')
print(f'Pre-ictal window: {window_start}s → {window_end}s')
print(f'Window duration : {window_end - window_start}s ({(window_end - window_start)/60:.1f} min)')

raw_preictal = raw.copy().crop(tmin=window_start, tmax=window_end)

print()
print('Pre-ictal segment loaded successfully.')

In [ ]:
# --- Visual comparison: raw vs filtered ---
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# Plot one channel before and after filtering for comparison
ch_idx = 0
ch_name = raw_preictal.ch_names[ch_idx]
times = raw_preictal.times[:int(raw_preictal.info['sfreq'] * 10)]  # first 10s
data_filtered = raw_preictal.get_data()[ch_idx, :len(times)]

axes[0].plot(times, data_filtered * 1e6, color='steelblue', linewidth=0.8)
axes[0].set_title(f'Filtered EEG — channel: {ch_name} (pre-ictal window, first 10s)')
axes[0].set_ylabel('Amplitude (µV)')

# Power spectral density
freqs, psd = welch(raw_preictal.get_data()[ch_idx], fs=raw_preictal.info['sfreq'], nperseg=512)
axes[1].semilogy(freqs[:80], psd[:80], color='darkorange', linewidth=1)
axes[1].set_title('Power Spectral Density (PSD)')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Power (V²/Hz)')
axes[1].axvspan(0.5, 4,   alpha=0.08, color='blue',   label='Delta')
axes[1].axvspan(4,   8,   alpha=0.08, color='green',  label='Theta')
axes[1].axvspan(8,   13,  alpha=0.08, color='orange', label='Alpha')
axes[1].axvspan(13,  30,  alpha=0.08, color='red',    label='Beta')
axes[1].axvspan(30,  40,  alpha=0.08, color='purple', label='Gamma')
axes[1].legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig('../experiments/results/psd_preictal.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to experiments/results/psd_preictal.png')

---
## 3. Band Power Computation

Using **Welch's method** to estimate average power in each EEG frequency band per channel.  
Bands: Delta (0.5–4 Hz), Theta (4–8 Hz), Alpha (8–13 Hz), Beta (13–30 Hz), Gamma (30–40 Hz)

In [ ]:
def band_power(data, sfreq, band):
    """
    Compute average power in a frequency band for each EEG channel.

    Parameters
    ----------
    data   : np.ndarray, shape (n_channels, n_times)
    sfreq  : float, sampling frequency in Hz
    band   : tuple, (low_freq, high_freq) in Hz

    Returns
    -------
    np.ndarray, shape (n_channels,) — mean power per channel in the band
    """
    freqs, psd = welch(data, fs=sfreq, nperseg=int(sfreq * 2))
    idx = np.logical_and(freqs >= band[0], freqs <= band[1])
    return np.mean(psd[:, idx], axis=1)

print('band_power() function defined.')

In [ ]:
# --- Compute band power for all channels ---
data  = raw_preictal.get_data()       # shape: (n_channels, n_times)
sfreq = raw_preictal.info['sfreq']

delta_power = band_power(data, sfreq, (0.5, 4))
theta_power = band_power(data, sfreq, (4,   8))
alpha_power = band_power(data, sfreq, (8,  13))
beta_power  = band_power(data, sfreq, (13, 30))
gamma_power = band_power(data, sfreq, (30, 40))

print('Band power computed for all channels.')
print(f'Shape per band: {delta_power.shape}  (one value per channel)')
print()
print(f'{"Channel":<20} {"Delta":>10} {"Theta":>10} {"Alpha":>10} {"Beta":>10} {"Gamma":>10}')
print('-' * 72)
for i, ch in enumerate(raw_preictal.ch_names):
    print(f'{ch:<20} {delta_power[i]:>10.2e} {theta_power[i]:>10.2e} '
          f'{alpha_power[i]:>10.2e} {beta_power[i]:>10.2e} {gamma_power[i]:>10.2e}')

---
## 4. Topographic Brain Maps

Projecting per-channel band power onto a 2D scalp map using the standard 10-20 montage.  
Color scale: **red = high power, blue = low power**.

In [ ]:
# --- Assign standard 10-20 electrode positions ---
# MNE needs to know where each electrode sits on the scalp
# CHB-MIT uses bipolar montage labels (e.g. 'FP1-F7') — we need to map
# to standard positions. We use the first electrode in each pair.

# Create a copy of info and set montage
info = raw_preictal.info.copy()

# Try to set the standard 10-20 montage; channels not in the montage are ignored
montage = mne.channels.make_standard_montage('standard_1020')
try:
    info.set_montage(montage, on_missing='ignore', verbose=False)
    print('Montage set successfully.')
except Exception as e:
    print(f'Montage note: {e}')
    print('Proceeding — channels without positions will be skipped in topomap.')

print(f'Channels with positions: {sum(1 for ch in info["chs"] if ch["loc"][:3].any())}')

In [ ]:
# --- Build position array manually from bipolar channel names ---
# CHB-MIT channels are bipolar (e.g. 'FP1-F7'). We extract the first
# electrode of each pair and look up its 2D position in the 10-20 montage.

montage = mne.channels.make_standard_montage('standard_1020')
montage_pos = {
    ch: montage.get_positions()['ch_pos'][ch]
    for ch in montage.get_positions()['ch_pos']
}

positions = []
valid_idx = []

for i, ch in enumerate(raw_preictal.ch_names):
    # Extract first electrode from bipolar pair (e.g. 'FP1-F7' → 'FP1')
    first_electrode = ch.split('-')[0].strip().upper()
    if first_electrode in montage_pos:
        xyz = montage_pos[first_electrode]
        positions.append(xyz[:2])   # take X and Y only (drop Z)
        valid_idx.append(i)

positions = np.array(positions)   # shape: (n_valid_channels, 2)
print(f'Channels with valid positions: {len(valid_idx)} / {len(raw_preictal.ch_names)}')

# --- Plot topographic maps using manual positions ---
bands = {
    'Delta\n(0.5–4 Hz)':  delta_power,
    'Theta\n(4–8 Hz)':    theta_power,
    'Alpha\n(8–13 Hz)':   alpha_power,
    'Beta\n(13–30 Hz)':   beta_power,
    'Gamma\n(30–40 Hz)':  gamma_power,
}

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
fig.suptitle(
    'Pre-ictal EEG Band Power — CHB01, Patient 1\n'
    '(60–30 minutes before seizure onset)',
    fontsize=13, y=1.02
)

last_im = None
for ax, (band_name, power) in zip(axes, bands.items()):
    # Keep only channels that had valid positions
    power_valid = power[valid_idx]
    power_norm = (power_valid - power_valid.min()) / (power_valid.max() - power_valid.min() + 1e-12)

    im, _ = mne.viz.plot_topomap(
        power_norm,
        positions,          # <-- pass coordinates directly, not info
        axes=ax,
        show=False,
        cmap='RdYlBu_r',
        vlim=(0, 1),
        contours=6,
        sphere=0.07
    )
    ax.set_title(band_name, fontsize=11, pad=10)
    last_im = im

cbar = fig.colorbar(last_im, ax=axes.tolist(), shrink=0.65, pad=0.02)
cbar.set_label('Normalized power (0 = min, 1 = max)', fontsize=10)

plt.tight_layout()
plt.savefig('../experiments/results/topomap_preictal_chb01.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

---
## 5. Observations

Write your findings here after running the notebook. This section becomes the basis for your paper's data analysis narrative.

**Template — fill in after running cells above:**

### Signal quality
- [ ] Were there visible artifacts in the raw signal? (eye blinks, muscle noise, electrode pop)
- [ ] Did filtering visibly clean the signal?
- [ ] Any channels that look noisy or flat (possibly disconnected)?

### Band power patterns
- [ ] Which frequency band showed the highest power during the pre-ictal window?
- [ ] Which brain region (frontal / temporal / parietal / occipital) was most active?
- [ ] Was the activity lateralized (left vs right hemisphere)?

### Clinical alignment
- [ ] CHB-MIT patients predominantly have **focal temporal lobe epilepsy** — did your maps show elevated temporal activity?
- [ ] Delta and theta elevation in temporal regions is a known pre-ictal signature — did you observe this?

### Next steps
- [ ] Repeat this analysis on inter-ictal windows (no seizure nearby) and compare maps
- [ ] Repeat across multiple patients to check cross-subject consistency
- [ ] Move validated preprocessing steps into `src/preprocessing.py`